# MATLAB partial-discharge data audit

**Objective.** Validate the observed MATLAB structure, labels, counts and source partitions.

**Inputs.** `engineering-partial-discharge-noise-signals@v1` from the local raw-data adapter.

**Outputs.** Schema audit, class counts, finite-value checks and a source-partition manifest.

**Experimental role.** Data contract validation before the confirmatory experiment.

**Leakage constraints.** Use `Tr1`, `Va1` and `Te1` only for this audit. Describe `Te2` from the registered local manifest; do not load it before protocol freeze.

In [ ]:
import numpy as np
from partial_discharge_adaptive_fusion.dataset import load_mat_partition, resolve_dataset
from partial_discharge_adaptive_fusion.splits import matlab_manifest

dataset = resolve_dataset('engineering-partial-discharge-noise-signals', 'v1')
batches = {name: load_mat_partition(dataset, name) for name in ('Tr1.mat', 'Va1.mat', 'Te1.mat')}
for name, batch in batches.items():
    assert batch.signal.shape[1] == 400
    assert not batch.metadata['sample_id'].duplicated().any()
    print(name, batch.signal.shape, np.unique(batch.label, return_counts=True), np.isfinite(batch.signal).all())
manifest = matlab_manifest({name: batch.label for name, batch in batches.items()}, dataset_id=dataset.dataset_id, dataset_version=dataset.version)
manifest.validate()
display(manifest.frame.groupby(['partition','split']).size().rename('n').reset_index())

## Findings and handoff

Confirm the `signals`/`labels` structure and preserve the original `Tr1/Va1/Te1/Te2` semantics. This notebook does not tune preprocessing or models.

**Next stage:** freeze the MATLAB protocol and keep `Te2` protected until all decisions are serialized.